In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.abspath("../.."))

In [3]:
from datalemur_utils import extract_iframe_from_url

In [4]:
url = "https://datalemur.com/questions/completed-trades"

In [5]:
extract_iframe_from_url(url)

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, sum as Sum

In [7]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/05 02:12:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
users = spark.read\
.option("delimiter", "\t")\
.option("header", "true")\
.option("inferSchema", "true")\
.format("csv")\
.load("users.txt")

trades = spark.read\
.option("delimiter", "\t")\
.option("header", "true")\
.option("inferSchema", "true")\
.format("csv")\
.load("trades.txt")

In [9]:
users.createOrReplaceTempView("users")
trades.createOrReplaceTempView("trades")

In [10]:
spark.sql(
    """
    select city, sum(case when status='Completed' then 1 else 0 end) as total_orders from users u join trades t on u.user_id=t.user_id
    group by 1 order by 2 desc limit 3
    """
).show()

+-------------+------------+
|         city|total_orders|
+-------------+------------+
|San Francisco|           3|
|       Boston|           2|
|       Denver|           1|
+-------------+------------+



In [11]:
users.join(trades, on="user_id", how="inner")\
.groupBy("city").agg(
    Sum(
        when(col("status")=="Completed", 1).otherwise(0)
    ).alias("total_orders")
)\
.orderBy(col("total_orders").desc())\
.limit(3)\
.show()

+-------------+------------+
|         city|total_orders|
+-------------+------------+
|San Francisco|           3|
|       Boston|           2|
|       Denver|           1|
+-------------+------------+



In [12]:
spark.stop()